# LeetCode #85: Maximal Rectangle

https://leetcode.com/problems/maximal-rectangle/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m^2 n^2)$ | $O(1)$ |
| **Optimal: Histogram Stack ★** | $O(mn)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
For every pair of rows, scan each column to find the tallest all-1 rectangle anchored between those rows. Quadratic over both dimensions.

### Optimal: Histogram Stack ★
Build a running height array: for each row, treat the number of consecutive 1s above (including the current row) as a bar height. Then solve "Largest Rectangle in Histogram" on that height array using a monotonic stack in $O(n)$. Repeating per row gives $O(mn)$ total.

**Constraints:**
* `rows == matrix.length`
* `cols == matrix[i].length`
* `1 <= rows, cols <= 200`
* `matrix[i][j]` is `'0'` or `'1'`

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaximalRectangle(char[][] matrix) {
        int m = matrix.Length, n = matrix[0].Length;
        int[] heights = new int[n];
        int maxArea = 0;

        for (int r = 0; r < m; r++) {
            // Build histogram: extend bar if '1', reset to 0 if '0'
            for (int c = 0; c < n; c++)
                heights[c] = matrix[r][c] == '1' ? heights[c] + 1 : 0;

            // Largest rectangle in the current histogram
            maxArea = Math.Max(maxArea, LargestInHistogram(heights));
        }
        return maxArea;
    }

    private int LargestInHistogram(int[] h) {
        // Monotonic increasing stack; stores column indices
        var stack = new Stack<int>();
        int maxA = 0, n = h.Length;

        for (int i = 0; i <= n; i++) {
            int cur = i == n ? 0 : h[i];
            // Pop bars taller than current — their right boundary is i
            while (stack.Count > 0 && h[stack.Peek()] > cur) {
                int height = h[stack.Pop()];
                // Left boundary is the new top of stack (or -1 if empty)
                int width = stack.Count == 0 ? i : i - stack.Peek() - 1;
                maxA = Math.Max(maxA, height * width);
            }
            stack.Push(i);
        }
        return maxA;
    }
}

### Python

In [ ]:
class Solution:
    def maximalRectangle(self, matrix: list[list[str]]) -> int:
        if not matrix:
            return 0
        n = len(matrix[0])
        heights = [0] * n
        max_area = 0

        for row in matrix:
            # Extend bar height if '1', reset to 0 for '0'
            for c in range(n):
                heights[c] = heights[c] + 1 if row[c] == '1' else 0

            max_area = max(max_area, self._largest_in_histogram(heights))

        return max_area

    def _largest_in_histogram(self, h: list[int]) -> int:
        # Monotonic increasing stack of indices
        stack: list[int] = []
        max_a = 0
        for i in range(len(h) + 1):
            cur = h[i] if i < len(h) else 0
            while stack and h[stack[-1]] > cur:
                height = h[stack.pop()]
                width = i if not stack else i - stack[-1] - 1
                max_a = max(max_a, height * width)
            stack.append(i)
        return max_a

### Go

In [ ]:
func maximalRectangle(matrix [][]byte) int {
    if len(matrix) == 0 {
        return 0
    }
    n := len(matrix[0])
    heights := make([]int, n)
    maxArea := 0

    for _, row := range matrix {
        // Extend bar height if '1', reset if '0'
        for c := 0; c < n; c++ {
            if row[c] == '1' {
                heights[c]++
            } else {
                heights[c] = 0
            }
        }
        maxArea = max(maxArea, largestInHistogram(heights))
    }
    return maxArea
}

func largestInHistogram(h []int) int {
    // Monotonic increasing stack of column indices
    stack := []int{}
    maxA := 0
    for i := 0; i <= len(h); i++ {
        cur := 0
        if i < len(h) {
            cur = h[i]
        }
        for len(stack) > 0 && h[stack[len(stack)-1]] > cur {
            height := h[stack[len(stack)-1]]
            stack = stack[:len(stack)-1]
            width := i
            if len(stack) > 0 {
                width = i - stack[len(stack)-1] - 1
            }
            if height*width > maxA {
                maxA = height * width
            }
        }
        stack = append(stack, i)
    }
    return maxA
}

func max(a, b int) int {
    if a > b { return a }
    return b
}

### Rust

In [ ]:
impl Solution {
    pub fn maximal_rectangle(matrix: Vec<Vec<char>>) -> i32 {
        if matrix.is_empty() { return 0; }
        let n = matrix[0].len();
        let mut heights = vec![0i32; n];
        let mut max_area = 0;

        for row in &matrix {
            // Extend bar height if '1', reset if '0'
            for c in 0..n {
                heights[c] = if row[c] == '1' { heights[c] + 1 } else { 0 };
            }
            max_area = max_area.max(Self::largest_in_histogram(&heights));
        }
        max_area
    }

    fn largest_in_histogram(h: &[i32]) -> i32 {
        // Monotonic increasing stack of indices
        let mut stack: Vec<usize> = Vec::new();
        let mut max_a = 0i32;
        for i in 0..=h.len() {
            let cur = if i < h.len() { h[i] } else { 0 };
            while let Some(&top) = stack.last() {
                if h[top] <= cur { break; }
                stack.pop();
                let width = if stack.is_empty() { i } else { i - stack[stack.len()-1] - 1 };
                max_a = max_a.max(h[top] * width as i32);
            }
            stack.push(i);
        }
        max_a
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `matrix = [["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]`
Row 2 yields heights `[3,1,3,2,2]`; the histogram's largest rectangle spans columns 2–4 with height 2, giving area 6.

### 2. Slightly Complex
**Input:** `matrix = [["0","1"],["1","0"]]`
Row 0 gives heights `[0,1]`, max area 1. Row 1 gives `[1,0]`, max area 1. Answer is 1.

### 3. Edge Case: Time Factor
**Input:** `200×200` matrix of all `'1'`s.
Every row updates a uniform height array; the histogram stack processes $n$ bars per row, so the $O(mn) = O(40000)$ scan completes quickly — no quadratic blowup.

### 4. Edge Case: Space Factor
**Input:** Single row of 200 `'1'`s.
Heights array is length 200 and the stack holds at most 200 indices, so space stays $O(n)$ regardless of the row count.

### 5. Almost-Impossible but Plausible
**Input:** `matrix = [["1","1","1","1"],["1","1","1","1"],["0","0","0","1"],["1","1","1","1"]]`
The break at row 2 resets columns 0–2 to height 0, leaving only column 3 with height 4. The optimal rectangle spans columns 0–3 in rows 0–1, area = $4 \times 2 = 8$.